# 🔐 Project 04 — Password Toolkit (Security-Aware Python)

**Skills used:** `secrets`, `string`, `math` (entropy), `hashlib` (PBKDF2), `hmac.compare_digest`, `re`, `functools`, classes, `getpass`, JSON. You'll also learn *why* certain choices are made — the difference between a toy and secure code.

> ⚠️ Educational project. For real systems use a vetted library (e.g. `argon2-cffi`, `bcrypt`) and a password manager for humans.

---
## 1. Generate strong passwords the right way
`random` is predictable (Mersenne Twister) — **never** use it for secrets. `secrets` uses the OS's cryptographically secure generator.

In [1]:
import secrets, string, math, re

def generate_password(length=16, upper=True, digits=True, symbols=True, avoid_ambiguous=True):
    pools = [string.ascii_lowercase]
    if upper:   pools.append(string.ascii_uppercase)
    if digits:  pools.append(string.digits)
    if symbols: pools.append("!@#$%^&*()-_=+[]{};:,.?")
    if avoid_ambiguous:
        pools = ["".join(c for c in p if c not in "O0oIl1|") for p in pools]
    if length < len(pools):
        raise ValueError("length too small for the chosen character classes")
    chars = [secrets.choice(p) for p in pools]                       # guarantee at least one of each class
    alphabet = "".join(pools)
    chars += [secrets.choice(alphabet) for _ in range(length - len(chars))]
    secrets.SystemRandom().shuffle(chars)                            # secure shuffle
    return "".join(chars)

for _ in range(3):
    print(generate_password())
print(generate_password(24, symbols=False))

def generate_passphrase(words=5, wordlist=None, sep="-"):
    wordlist = wordlist or ["orbit", "maple", "quartz", "lantern", "velvet", "harbor", "cobalt", "meadow",
                            "ember", "glacier", "falcon", "pixel", "saffron", "tundra", "whisper", "zenith"]
    return sep.join(secrets.choice(wordlist) for _ in range(words))
print(generate_passphrase())

2EMd,Pmu8h*mD8-,
QY=7;7M):f+=SX;W
+6z]T4bCt,J=JPF8
p5W8zpk72sVYeRnwEU8zuAg6
maple-maple-glacier-meadow-glacier


## 2. Measure strength: entropy
**Entropy** (bits) = `log2(alphabet_size ** length)` = `length × log2(alphabet_size)`. Each extra bit doubles an attacker's work. Rough guide: < 40 weak · 60 OK · 80+ strong · 100+ excellent.

In [2]:
def alphabet_size(pw):
    size = 0
    if re.search(r"[a-z]", pw): size += 26
    if re.search(r"[A-Z]", pw): size += 26
    if re.search(r"\d", pw):    size += 10
    if re.search(r"[^\w\s]", pw): size += 32
    if re.search(r"\s", pw):    size += 1
    return max(size, 1)

COMMON = {"password", "123456", "12345678", "qwerty", "letmein", "admin", "welcome", "iloveyou", "abc123", "monkey", "dragon"}

def analyse(pw):
    bits = len(pw) * math.log2(alphabet_size(pw))
    problems = []
    if pw.lower() in COMMON:                        problems.append("is a very common password")
    if re.search(r"(.)\1{2,}", pw):                 problems.append("has 3+ repeated characters in a row")
    if re.search(r"(?:012|123|234|345|456|567|678|789|abc|bcd|cde|def)", pw.lower()): problems.append("contains an obvious sequence")
    if len(pw) < 12:                                problems.append("shorter than 12 characters")
    if pw.isalpha() or pw.isdigit():                problems.append("uses only one character class")
    if problems: bits *= 0.5                        # heavy penalty for known bad patterns
    label = ("very weak" if bits < 28 else "weak" if bits < 40 else "fair" if bits < 60
             else "strong" if bits < 80 else "excellent")
    return {"entropy_bits": round(bits, 1), "rating": label, "problems": problems}

for pw in ["password", "Tr0ub4dor&3", "correct-horse-battery-staple", generate_password(16), "aaaa1111"]:
    print(f"{pw[:32]:<34}", analyse(pw))

password                           {'entropy_bits': 18.8, 'rating': 'very weak', 'problems': ['is a very common password', 'shorter than 12 characters', 'uses only one character class']}
Tr0ub4dor&3                        {'entropy_bits': 36.1, 'rating': 'weak', 'problems': ['shorter than 12 characters']}
correct-horse-battery-staple       {'entropy_bits': 164.0, 'rating': 'excellent', 'problems': []}
vw?LKg]3FHC=T,c[                   {'entropy_bits': 104.9, 'rating': 'excellent', 'problems': []}
aaaa1111                           {'entropy_bits': 20.7, 'rating': 'very weak', 'problems': ['has 3+ repeated characters in a row', 'shorter than 12 characters']}


> ⚠️ **Read the number with care.** This estimator assumes every character was chosen *randomly*. That is true for `generate_password()` output but **not** for human-picked words: `correct-horse-battery-staple` scores ~164 "bits" here, but an attacker who guesses word combinations faces far less (four random common words ≈ 44 bits). Real strength checkers (like `zxcvbn`) model dictionaries, keyboard patterns and leaked-password lists. Treat this as a teaching model, not a verdict.

### How long to crack? (offline attack estimate)

In [3]:
def crack_time(bits, guesses_per_second=1e10):
    seconds = 2 ** bits / 2 / guesses_per_second          # on average you find it halfway
    for unit, div in [("years", 31_557_600), ("days", 86_400), ("hours", 3600), ("minutes", 60), ("seconds", 1)]:
        if seconds >= div:
            return f"{seconds / div:,.1f} {unit}"
    return "instantly"

for bits in (30, 50, 70, 90, 110):
    print(f"{bits:>4} bits  ->  {crack_time(bits)}")

  30 bits  ->  instantly
  50 bits  ->  15.6 hours
  70 bits  ->  1,870.5 years
  90 bits  ->  1,961,397,633.7 years
 110 bits  ->  2,056,674,485,121,978.5 years


## 3. Store passwords safely: salted, slow hashing
**Never** store plain text, or a fast unsalted hash (MD5/SHA-256 of the password): attackers use rainbow tables and GPUs.
Use a **salt** (unique random bytes per user) and a **slow key-derivation function** (PBKDF2, scrypt, bcrypt, Argon2).

In [4]:
import hashlib, hmac, os, base64

ITERATIONS = 200_000

def hash_password(password, iterations=ITERATIONS):
    salt = os.urandom(16)
    digest = hashlib.pbkdf2_hmac("sha256", password.encode(), salt, iterations)
    return f"pbkdf2_sha256${iterations}${base64.b64encode(salt).decode()}${base64.b64encode(digest).decode()}"

def verify_password(password, stored):
    scheme, iters, salt_b64, digest_b64 = stored.split("$")
    salt, expected = base64.b64decode(salt_b64), base64.b64decode(digest_b64)
    actual = hashlib.pbkdf2_hmac("sha256", password.encode(), salt, int(iters))
    return hmac.compare_digest(actual, expected)            # constant-time comparison: defeats timing attacks

stored = hash_password("S3cret!pass")
print(stored)
print(verify_password("S3cret!pass", stored), verify_password("wrong", stored))
print("same password, different salt -> different hashes:", hash_password("S3cret!pass") != stored)

# Why the naive way is terrible:
print(hashlib.sha256(b"password").hexdigest())     # identical for every user, instantly reversible via lookup tables

pbkdf2_sha256$200000$PnAVNfMZ7KrIj4BCn63kEQ==$skuWyrlKDRwOwta16G3KiuuJ3wbfj/+kcUn8hP5mLGE=
True False
same password, different salt -> different hashes: True
5e884898da28047151d0e56f8dc6292773603d0d6aabbdd62a11ef721d1542d8


## 4. A tiny user store with registration + login + lockout

In [5]:
import json, time

class AuthStore:
    MAX_FAILS = 3
    LOCK_SECONDS = 30

    def __init__(self):
        self.users = {}                                    # username -> {"hash": ..., "fails": 0, "locked_until": 0}

    def register(self, username, password):
        if username in self.users:
            raise ValueError("username taken")
        info = analyse(password)
        if info["rating"] in ("very weak", "weak"):
            raise ValueError(f"password too weak ({info['rating']}): {', '.join(info['problems']) or 'increase length'}")
        self.users[username] = {"hash": hash_password(password), "fails": 0, "locked_until": 0}

    def login(self, username, password, now=None):
        now = time.time() if now is None else now
        user = self.users.get(username)
        if user is None:
            hash_password("dummy")                         # do similar work to hide whether the user exists
            return False, "invalid credentials"
        if now < user["locked_until"]:
            return False, f"account locked for {int(user['locked_until'] - now)}s"
        if verify_password(password, user["hash"]):
            user["fails"] = 0
            return True, "welcome"
        user["fails"] += 1
        if user["fails"] >= self.MAX_FAILS:
            user["locked_until"] = now + self.LOCK_SECONDS
            user["fails"] = 0
        return False, "invalid credentials"

store = AuthStore()
for pw in ("password", "abc12345"):
    try:
        store.register("asha", pw)
    except ValueError as e:
        print("register rejected:", e)
store.register("asha", "Blue!Lantern-92-Tundra")
print(store.login("asha", "wrong1", now=1000))
print(store.login("asha", "wrong2", now=1001))
print(store.login("asha", "wrong3", now=1002))
print("during lockout ->", store.login("asha", "Blue!Lantern-92-Tundra", now=1010))
print("after lockout  ->", store.login("asha", "Blue!Lantern-92-Tundra", now=1040))
print("unknown user   ->", store.login("ghost", "x"))

register rejected: password too weak (very weak): is a very common password, shorter than 12 characters, uses only one character class
register rejected: password too weak (very weak): contains an obvious sequence, shorter than 12 characters
(False, 'invalid credentials')
(False, 'invalid credentials')
(False, 'invalid credentials')
during lockout -> (False, 'account locked for 22s')
after lockout  -> (True, 'welcome')
unknown user   -> (False, 'invalid credentials')


## 5. A vault: encrypt data with a password (concept)
Real encryption needs a vetted library. With `pip install cryptography` you'd derive a key from the password (PBKDF2/scrypt) and use **Fernet** (AES + HMAC):

```python
from cryptography.fernet import Fernet
import base64, hashlib, os

def key_from_password(password: str, salt: bytes) -> bytes:
    raw = hashlib.pbkdf2_hmac("sha256", password.encode(), salt, 300_000, dklen=32)
    return base64.urlsafe_b64encode(raw)

salt = os.urandom(16)
token = Fernet(key_from_password("master-pw", salt)).encrypt(b"my secret note")
plain = Fernet(key_from_password("master-pw", salt)).decrypt(token)
```
**Rules:** never invent your own crypto; never reuse salts/nonces; store `salt + token`, never the password or key.

## 6. Interactive command-line tool
`getpass` hides typing. Run this in a terminal (marked so it doesn't block the notebook).

In [ ]:
# notest
from getpass import getpass

def cli():
    print("1) generate  2) check strength  3) quit")
    while True:
        choice = input("> ").strip()
        if choice == "1":
            length = int(input("length [16]: ") or 16)
            print(generate_password(length))
        elif choice == "2":
            print(analyse(getpass("Password (hidden): ")))
        else:
            break

# cli()

---
## 🧠 Security lessons in one table
| Mistake | Why it's bad | Do this |
|---|---|---|
| `random.choice` for passwords/tokens | predictable | `secrets` |
| Storing plain text | one leak = total compromise | salted slow hash |
| Fast hash (MD5/SHA-256) | GPUs try billions/s | PBKDF2/scrypt/bcrypt/Argon2 |
| No salt | rainbow tables, same hash for same password | unique random salt per user |
| `==` on secrets | timing side channel | `hmac.compare_digest` |
| String-formatted SQL | SQL injection | parameterised queries |
| Secrets in source code | leak via git | environment variables / secret manager |
| Rolling your own crypto | subtle fatal flaws | `cryptography` library |

## 🚀 Extensions
1. Check passwords against the **Have I Been Pwned** k-anonymity API (send only the first 5 chars of the SHA-1).
2. Add TOTP two-factor codes (`pyotp`).
3. Persist `AuthStore` to SQLite (Project 03) with a `users` table.
4. Build a GUI or Streamlit front-end.